# Machine Learning — Lab 2
## Exploratory Data Analysis and Data Quality

**Main Course Learning Outcome — CLO2**  
**CLO2:** Analyze datasets and apply appropriate preprocessing, transformation, and feature engineering techniques.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scipy`  
**Submission:** completed notebook containing predictions, exploratory analysis, manual calculations, cleaning decisions, debugging evidence, and reflection.

> **Assessment principle:** Generating plots is not enough. Full credit requires explaining **what the data show, what may be wrong, what should be cleaned, and why a particular treatment is justified**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Dataset orientation | 10 min | Identify observations, variables, target, and feature types |
| 2. Descriptive statistics | 20 min | Analyze center, spread, quartiles, skew, and outliers |
| 3. Visual EDA | 25 min | Use histograms, boxplots, and scatter plots |
| 4. Data-quality diagnosis | 30 min | Find missing values, duplicates, invalid values, and inconsistent categories |
| 5. Categorical association | 20 min | Build a contingency table and perform a Chi-square analysis |
| 6. Challenge, reflection & viva | 15 min | Defend cleaning decisions and interpret results |

> **Main idea:** Exploratory Data Analysis is not a search for attractive plots. It is a structured investigation of whether the data are **valid, representative, consistent, and suitable for learning**.

## Learning Objectives

By the end of this lab, you should be able to:

1. identify observations, features, targets, and variable types;
2. compute and interpret mean, median, standard deviation, quartiles, and IQR;
3. compare robust and non-robust summaries;
4. create and interpret histograms, boxplots, and scatter plots;
5. detect missing values, duplicate records, invalid entries, and inconsistent categories;
6. distinguish an outlier from an obvious data-entry error;
7. justify whether to keep, correct, cap, transform, or remove suspicious observations;
8. build a contingency table for two categorical variables;
9. compute expected counts and the Chi-square statistic;
10. explain what EDA reveals that a single model score cannot.

# Part I — Create and Inspect the Dataset

The notebook uses a **synthetic student-learning dataset** created locally so that every student can work without internet access.

The dataset deliberately contains several realistic quality problems:

- missing values;
- inconsistent category labels;
- duplicate rows;
- implausible values;
- extreme but potentially real observations.

Your job is to detect them rather than being told where they are.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Machine Learning Lab 2 environment ready.")

In [ ]:
# Create a reproducible master dataset.
rng = np.random.default_rng(3452)
n = 220

student_id = np.arange(10001, 10001 + n)

study_method = rng.choice(
    ["Individual", "Group", "Online"],
    size=n,
    p=[0.40, 0.32, 0.28]
)

study_hours = np.clip(rng.normal(8.0, 3.0, size=n), 0.5, 18.0)
attendance_pct = np.clip(rng.normal(82.0, 11.0, size=n), 35.0, 100.0)
previous_gpa = np.clip(rng.normal(2.9, 0.55, size=n), 1.2, 4.0)
quiz_score = np.clip(rng.normal(72.0, 13.0, size=n), 20.0, 100.0)
lab_submissions = np.clip(np.rint(rng.normal(8.0, 2.0, size=n)), 0, 12).astype(int)

# A latent score used only to create a plausible performance category.
method_effect = np.select(
    [
        study_method == "Group",
        study_method == "Online",
    ],
    [4.0, -2.0],
    default=0.0,
)

latent = (
    1.8 * study_hours
    + 0.30 * attendance_pct
    + 8.0 * previous_gpa
    + 0.22 * quiz_score
    + 1.5 * lab_submissions
    + method_effect
    + rng.normal(0, 8, size=n)
)

performance_level = np.where(latent >= np.median(latent), "High", "Low")

df_master = pd.DataFrame({
    "student_id": student_id,
    "study_method": study_method,
    "study_hours": np.round(study_hours, 1),
    "attendance_pct": np.round(attendance_pct, 1),
    "previous_gpa": np.round(previous_gpa, 2),
    "quiz_score": np.round(quiz_score, 1),
    "lab_submissions": lab_submissions,
    "performance_level": performance_level,
})

# Inject realistic data-quality issues.
missing_att = rng.choice(df_master.index, size=10, replace=False)
missing_gpa = rng.choice(df_master.index.difference(missing_att), size=7, replace=False)
df_master.loc[missing_att, "attendance_pct"] = np.nan
df_master.loc[missing_gpa, "previous_gpa"] = np.nan

# Inconsistent categories.
category_rows = rng.choice(df_master.index, size=9, replace=False)
replacements = ["individual", "GROUP", "online ", "Group ", "INDIVIDUAL", "Online", "group", " Individual", "ONLINE"]
for idx, value in zip(category_rows, replacements):
    df_master.loc[idx, "study_method"] = value

# Implausible values.
invalid_rows = rng.choice(df_master.index, size=4, replace=False)
df_master.loc[invalid_rows[0], "attendance_pct"] = 135.0
df_master.loc[invalid_rows[1], "quiz_score"] = -8.0
df_master.loc[invalid_rows[2], "previous_gpa"] = 4.8
df_master.loc[invalid_rows[3], "study_hours"] = -3.0

# Extreme but potentially real values.
outlier_rows = rng.choice(df_master.index, size=3, replace=False)
df_master.loc[outlier_rows[0], "study_hours"] = 28.0
df_master.loc[outlier_rows[1], "study_hours"] = 31.0
df_master.loc[outlier_rows[2], "quiz_score"] = 100.0

# Add exact duplicate rows.
duplicate_source_rows = rng.choice(df_master.index, size=4, replace=False)
df_master = pd.concat([df_master, df_master.loc[duplicate_source_rows]], ignore_index=True)

print("Master dataset created.")
print("Shape:", df_master.shape)

## Task 1.1 — Personalized Working Sample

Enter the **last four digits** of your student ID.

Your ID determines a reproducible sample of 180 rows from the master dataset. This means different students may see different combinations of suspicious records.

Do not use another student's value.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID before continuing.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 2000 + (STUDENT_ID_LAST4 % 8000)

df = df_master.sample(
    n=180,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

print("Your EDA seed:", SEED)
print("Your dataset shape:", df.shape)

## Task 1.2 — First Inspection: Predict Before Running

Before running the next cell, write down what you expect to learn from:

- `df.head()`
- `df.info()`
- `df.describe()`

Then predict:

1. Which columns are numerical?
2. Which are categorical?
3. Which column is an identifier?
4. Which column could act as a target in a later supervised-learning task?
5. Which columns do you suspect may contain quality problems?

**Your prediction:**

In [ ]:
display(df.head(8))

print("\nDataFrame information:")
df.info()

print("\nNumerical summary:")
display(df.describe())

## Task 1.3 — Dataset Anatomy

Complete the table.

| Column | Role | Data type | Should normally be used as an input feature? | Reason |
|---|---|---|---|---|
| `student_id` |  |  |  |  |
| `study_method` |  |  |  |  |
| `study_hours` |  |  |  |  |
| `attendance_pct` |  |  |  |  |
| `previous_gpa` |  |  |  |  |
| `quiz_score` |  |  |  |  |
| `lab_submissions` |  |  |  |  |
| `performance_level` |  |  |  |  |

Answer also:

1. What does one row represent?
2. What does one column represent?
3. Why is an identifier such as `student_id` usually a poor predictive feature?

# Part II — Descriptive Statistics

A numerical feature can be summarized by:

- center;
- spread;
- quantiles;
- minimum/maximum;
- shape.

However, one summary statistic rarely tells the whole story.

## Task 2.1 — Mean vs. Median

Choose one numerical feature that may contain extreme values.

Before running the next cell:

1. Predict whether its **mean** or **median** will be more affected by extreme values.
2. Predict which statistic better represents a typical observation.
3. Explain why.

In [ ]:
numerical_cols = [
    "study_hours",
    "attendance_pct",
    "previous_gpa",
    "quiz_score",
    "lab_submissions",
]

summary = pd.DataFrame({
    "mean": df[numerical_cols].mean(),
    "median": df[numerical_cols].median(),
    "std": df[numerical_cols].std(),
    "min": df[numerical_cols].min(),
    "max": df[numerical_cols].max(),
})

display(summary.round(3))

## Task 2.2 — Interpret Mean and Median

For **two** numerical features:

1. record the mean;
2. record the median;
3. compare them;
4. explain whether the difference suggests skew, outliers, or possible data-quality problems.

| Feature | Mean | Median | Interpretation |
|---|---:|---:|---|
|  |  |  |  |
|  |  |  |  |

## Task 2.3 — Quartiles and IQR

For a numerical feature $x$:

$$
IQR = Q_3 - Q_1.
$$

A common outlier rule is:

$$
x < Q_1 - 1.5IQR
$$

or

$$
x > Q_3 + 1.5IQR.
$$

Complete the function below so that it returns:

- $Q_1$;
- $Q_3$;
- IQR;
- lower fence;
- upper fence.

Do not remove values yet.

In [ ]:
def iqr_summary(series):
    clean = series.dropna()

    # TODO: compute Q1 and Q3.
    q1 = None
    q3 = None

    # TODO: compute IQR and fences.
    iqr = None
    lower = None
    upper = None

    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_fence": lower,
        "upper_fence": upper,
    }

### Self-check

After completing the function, run this cell.

In [ ]:
toy = pd.Series([1, 2, 3, 4, 5, 6, 7, 8])
check = iqr_summary(toy)

assert abs(check["Q1"] - 2.75) < 1e-9
assert abs(check["Q3"] - 6.25) < 1e-9
assert abs(check["IQR"] - 3.5) < 1e-9

print("IQR function tests passed.")

In [ ]:
for col in ["study_hours", "attendance_pct", "previous_gpa", "quiz_score"]:
    print(col, iqr_summary(df[col]))

## Task 2.4 — Outlier Rule vs. Domain Validity

For each suspicious extreme value found by the IQR rule, ask:

1. Is it mathematically unusual?
2. Is it physically/academically possible?
3. Is it likely a data-entry error?
4. Could it be rare but real?
5. Should it be removed automatically?

Choose at least **two** suspicious values and justify a treatment decision.

| Feature/value | IQR flag? | Valid in domain? | Your treatment | Justification |
|---|---|---|---|---|
|  |  |  |  |  |
|  |  |  |  |  |

# Part III — Visual Exploratory Data Analysis

Plots help reveal patterns that summary tables can hide.

You will use:

- histograms for distribution shape;
- boxplots for quartiles and extreme values;
- scatter plots for relationships.

Do not describe a plot using only “high” or “low”. Explain the **shape, concentration, spread, unusual points, and possible implication**.

## Task 3.1 — Histogram

Before running the plot:

1. Which numerical feature do you expect to be most skewed?
2. Which feature may show implausible values?
3. What pattern would make you prefer median over mean?

Write your prediction first.

In [ ]:
feature = "study_hours"

plt.figure(figsize=(7, 4))
plt.hist(df[feature].dropna(), bins=16, edgecolor="black")
plt.xlabel(feature)
plt.ylabel("Frequency")
plt.title(f"Histogram of {feature}")
plt.show()

## Task 3.2 — Interpret the Histogram

For `study_hours`, describe:

- where most values are concentrated;
- whether the distribution appears symmetric or skewed;
- whether any values appear isolated;
- whether isolated values are necessarily errors;
- which statistic, mean or median, better describes a typical student in your sample.

**Your interpretation:**

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.boxplot(df["study_hours"].dropna(), vert=False)
plt.xlabel("Study hours")
plt.title("Boxplot of Study Hours")
plt.show()

## Task 3.3 — Boxplot Interpretation

Use the boxplot and your IQR calculation together.

Answer:

1. What does the box represent?
2. What does the line inside the box represent?
3. Which values are flagged as potential outliers?
4. Why must you still inspect the domain before removing them?

## Task 3.4 — Scatter Plot

Examine the relationship between study hours and quiz score.

Before running the next cell:

- predict whether the association should be positive, negative, or weak;
- predict whether one unusual point could distort interpretation.

In [ ]:
plot_df = df[["study_hours", "quiz_score", "performance_level"]].dropna()

plt.figure(figsize=(7, 5))
plt.scatter(plot_df["study_hours"], plot_df["quiz_score"], alpha=0.75)
plt.xlabel("Study Hours")
plt.ylabel("Quiz Score")
plt.title("Study Hours vs. Quiz Score")
plt.show()

## Task 3.5 — Interpret the Scatter Plot

Answer:

1. Is there an obvious linear relationship?
2. Are there isolated observations?
3. Could an invalid negative value affect correlation?
4. Does association imply causation?
5. Name one unobserved factor that could affect both study hours and quiz score.

**Your answers:**

In [ ]:
corr = df[["study_hours", "attendance_pct", "previous_gpa", "quiz_score", "lab_submissions"]].corr(numeric_only=True)
display(corr.round(3))

## Task 3.6 — Correlation Reasoning

Choose the strongest positive correlation in the matrix.

Answer:

1. Which two variables are involved?
2. What does the sign mean?
3. What does the magnitude mean?
4. Why does this not prove that one variable causes the other?
5. Could invalid or extreme values influence the coefficient?

**Your answers:**

# Part IV — Data Quality Diagnosis

A useful quality audit examines at least:

$$
\text{Missingness}
+\text{Duplicates}
+\text{Validity}
+\text{Consistency}
+\text{Extremes}.
$$

You will diagnose each category separately before deciding how to clean.

## Task 4.1 — Missing Values

Predict which columns may contain missing values, then run the audit.

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": 100 * df.isna().mean()
}).sort_values("missing_count", ascending=False)

display(missing_report.round(2))

## Task 4.2 — Missingness Decisions

For every column with missing values:

1. state how many values are missing;
2. calculate the percentage;
3. decide whether you would:
   - remove rows;
   - impute;
   - create a missingness indicator;
   - investigate the data source first;
4. justify your decision.

> You are not required to impute in this lab. The important skill is **reasoned diagnosis**.

## Task 4.3 — Duplicate Records

Before running the next cell:

1. What is an exact duplicate?
2. Why is a repeated student record suspicious in this dataset?
3. Could duplicate transactions be valid in another dataset?

**Your prediction:**

In [ ]:
duplicate_mask = df.duplicated(keep=False)

print("Number of rows participating in exact duplicates:", duplicate_mask.sum())
display(df.loc[duplicate_mask].sort_values("student_id"))

## Task 4.4 — Duplicate Decision

Answer:

1. How many exact duplicate rows are in your personalized sample?
2. Why would keeping duplicate student records distort some analyses?
3. Would you remove every duplicated `student_id` automatically?
4. What additional information would you want before deleting records?

**Your answers:**

## Task 4.5 — Invalid Numerical Values

Define sensible domain rules:

- `study_hours` should be between 0 and 24 hours/day;
- `attendance_pct` should be between 0 and 100;
- `previous_gpa` should be between 0 and 4;
- `quiz_score` should be between 0 and 100;
- `lab_submissions` should be between 0 and 12.

Complete the validation function.

In [ ]:
def invalid_numeric_rows(data):
    rules = {
        "study_hours": (0, 24),
        "attendance_pct": (0, 100),
        "previous_gpa": (0, 4),
        "quiz_score": (0, 100),
        "lab_submissions": (0, 12),
    }

    problems = []

    for col, (low, high) in rules.items():
        # TODO: identify non-missing values outside [low, high]
        bad_mask = None

        if bad_mask is None:
            raise NotImplementedError("Complete bad_mask logic.")

        for idx in data.index[bad_mask]:
            problems.append({
                "row": int(idx),
                "student_id": int(data.loc[idx, "student_id"]),
                "feature": col,
                "value": data.loc[idx, col],
                "valid_range": f"[{low}, {high}]",
            })

    return pd.DataFrame(problems)

### Self-check

After completing the function, run:

In [ ]:
toy_invalid = pd.DataFrame({
    "student_id": [1, 2],
    "study_hours": [5, -1],
    "attendance_pct": [90, 101],
    "previous_gpa": [3.0, 4.2],
    "quiz_score": [80, 50],
    "lab_submissions": [8, 9],
})

toy_problems = invalid_numeric_rows(toy_invalid)
assert set(toy_problems["feature"]) == {"study_hours", "attendance_pct", "previous_gpa"}

print("Numeric validation tests passed.")

In [ ]:
numeric_problems = invalid_numeric_rows(df)
display(numeric_problems)

## Task 4.6 — Invalid vs. Extreme

For each detected invalid value:

- explain why it violates the domain;
- propose a treatment: correction, set to missing, remove row, or investigate.

Then compare with an extreme but valid value such as `study_hours = 24` or a perfect `quiz_score = 100`.

Why should those cases be treated differently?

**Your answer:**

## Task 4.7 — Inconsistent Categories

Predict how many unique representations of the three intended study methods may appear.

Then run:

In [ ]:
print("Raw unique study_method values:")
for value in sorted(df["study_method"].dropna().astype(str).unique()):
    print(repr(value))

print("\nValue counts:")
display(df["study_method"].value_counts(dropna=False))

## Task 4.8 — Standardize the Categories

The intended categories are:

```text
Individual
Group
Online
```

Complete the function so that variations in capitalization and whitespace are standardized.

Example:

```text
" group " -> "Group"
"INDIVIDUAL" -> "Individual"
"online " -> "Online"
```

In [ ]:
def clean_study_method(series):
    # TODO:
    # 1. convert values to strings;
    # 2. strip whitespace;
    # 3. normalize capitalization.
    cleaned = None
    return cleaned

In [ ]:
toy_methods = pd.Series([" group ", "INDIVIDUAL", "online ", "Group"])
toy_cleaned = clean_study_method(toy_methods)

assert toy_cleaned.tolist() == ["Group", "Individual", "Online", "Group"]

print("Category-cleaning tests passed.")

In [ ]:
df_clean_categories = df.copy()
df_clean_categories["study_method"] = clean_study_method(df_clean_categories["study_method"])

print("Clean unique values:")
print(sorted(df_clean_categories["study_method"].dropna().unique()))

display(df_clean_categories["study_method"].value_counts())

## Task 4.9 — Why Category Cleaning Matters

Answer:

1. How many raw category labels did your sample contain before cleaning?
2. How many intended categories remain after cleaning?
3. What would happen if one-hot encoding were applied before cleaning?
4. Why is category standardization a modeling issue rather than only a presentation issue?

# Part V — Categorical Association with a Chi-Square Test

We now examine whether two categorical variables appear statistically associated:

- `study_method`
- `performance_level`

The question is:

> Is the observed performance distribution similar across study methods, or is the difference larger than we would expect under independence?

Use the cleaned study-method categories.

In [ ]:
analysis_df = df_clean_categories[["study_method", "performance_level"]].dropna()

observed = pd.crosstab(
    analysis_df["study_method"],
    analysis_df["performance_level"]
)

print("Observed contingency table:")
display(observed)

## Task 5.1 — State the Hypotheses

Write:

- **Null hypothesis $H_0$**
- **Alternative hypothesis $H_1$**

Use the variables in this lab rather than generic wording.

**Your answer:**

## Task 5.2 — Expected Counts

If two variables are independent, the expected count in cell $(i,j)$ is:

$$
E_{ij}
=
\frac{(\text{row total}_i)(\text{column total}_j)}
{\text{grand total}}.
$$

Before running any verification library, compute **one expected count manually** from your observed table.

Show:

- row total;
- column total;
- grand total;
- expected count.

## Task 5.3 — Complete the Expected-Count Function

Complete the function below.

In [ ]:
def expected_counts_from_table(table):
    values = table.to_numpy(dtype=float)

    row_totals = values.sum(axis=1, keepdims=True)
    col_totals = values.sum(axis=0, keepdims=True)
    grand_total = values.sum()

    # TODO: compute the full expected-count matrix.
    expected = None

    if expected is None:
        raise NotImplementedError("Complete the expected-count calculation.")

    return pd.DataFrame(
        expected,
        index=table.index,
        columns=table.columns
    )

In [ ]:
toy_table = pd.DataFrame(
    [[20, 30], [10, 40]],
    index=["A", "B"],
    columns=["Low", "High"]
)

toy_expected = expected_counts_from_table(toy_table)

assert toy_expected.shape == toy_table.shape
assert abs(toy_expected.values.sum() - toy_table.values.sum()) < 1e-9

print("Expected-count function basic test passed.")
display(toy_expected)

In [ ]:
expected_manual = expected_counts_from_table(observed)
print("Expected counts under independence:")
display(expected_manual.round(3))

## Task 5.4 — Compute the Chi-Square Statistic

The statistic is:

$$
\chi^2
=
\sum_{i,j}
\frac{(O_{ij}-E_{ij})^2}{E_{ij}}.
$$

Complete the function.

In [ ]:
def chi_square_statistic(observed_table, expected_table):
    O = observed_table.to_numpy(dtype=float)
    E = expected_table.to_numpy(dtype=float)

    # TODO: implement the formula above.
    statistic = None

    if statistic is None:
        raise NotImplementedError("Complete the Chi-square statistic.")

    return float(statistic)

In [ ]:
chi2_manual = chi_square_statistic(observed, expected_manual)
print(f"Manual Chi-square statistic: {chi2_manual:.4f}")

## Task 5.5 — Degrees of Freedom

For a contingency table with $r$ rows and $c$ columns:

$$
df=(r-1)(c-1).
$$

Calculate the degrees of freedom for your table **before** running the verification below.

**Your calculation:**

In [ ]:
chi2_lib, p_value, dof, expected_lib = chi2_contingency(observed)

print(f"SciPy Chi-square statistic: {chi2_lib:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.6f}")

print("\nDifference between manual and SciPy statistics:",
      abs(chi2_manual - chi2_lib))

## Task 5.6 — Interpret the Test

Using significance level:

$$
\alpha=0.05,
$$

answer:

1. Is $p<0.05$?
2. Would you reject or fail to reject $H_0$?
3. Does this prove that study method **causes** performance level?
4. Name at least two possible confounding variables already present in the dataset.
5. Why should a statistical association not automatically become a predictive feature decision?

**Your interpretation:**

# Part VI — Build a Data-Quality Report

Before preprocessing in the next lab, summarize what should be fixed.

Create a report with at least these issue types:

- missing values;
- duplicates;
- invalid numerical values;
- inconsistent categories;
- potential outliers.

For each issue, state:

1. evidence;
2. proposed action;
3. justification.

## Task 6.1 — Your Data-Quality Report

Complete:

| Issue | Evidence from your sample | Proposed treatment | Justification |
|---|---|---|---|
| Missing values |  |  |  |
| Duplicates |  |  |  |
| Invalid numerical values |  |  |  |
| Inconsistent categories |  |  |  |
| Potential outliers |  |  |  |

> Do **not** write “remove all bad rows.” Different problems require different treatments.

## Task 6.2 — Deliberate Debugging

The code below tries to remove invalid attendance values:

```python
bad = df["attendance_pct"] < 0 & df["attendance_pct"] > 100
```

It is wrong in two ways.

Explain:

1. the logical error;
2. the operator-precedence/readability problem;
3. the correct Boolean condition.

Then write the corrected one-line expression below.

In [ ]:
# TODO: Write the correct Boolean mask.
bad_attendance_mask = None

In [ ]:
if bad_attendance_mask is None:
    raise ValueError("Complete bad_attendance_mask.")

# Valid missing values are not treated as invalid here.
expected_mask = (
    df["attendance_pct"].notna()
    & ((df["attendance_pct"] < 0) | (df["attendance_pct"] > 100))
)

assert bad_attendance_mask.equals(expected_mask)
print("Attendance validation logic is correct.")

# Part VII — Personalized Analysis Challenge

Your student-ID seed assigns one numerical feature for deeper analysis.

Run the next cell.

In [ ]:
challenge_features = [
    "study_hours",
    "attendance_pct",
    "previous_gpa",
    "quiz_score",
]

assigned_feature = challenge_features[SEED % len(challenge_features)]
print("Your assigned feature:", assigned_feature)

## Task 7.1 — Analyze Your Assigned Feature

For the assigned feature, report:

1. count of non-missing values;
2. missing count;
3. mean;
4. median;
5. standard deviation;
6. $Q_1$;
7. $Q_3$;
8. IQR;
9. IQR lower and upper fences;
10. minimum and maximum;
11. whether any value violates the domain;
12. one histogram or boxplot interpretation;
13. your recommended treatment before modeling.

You may reuse functions from earlier cells.

**Your analysis:**

## Task 7.2 — Individual Understanding Check

Your instructor may select **one** of the following for a 60–90 second explanation.

1. Why can the mean and median tell different stories?
2. Why is an IQR outlier not automatically an error?
3. Explain the difference between a duplicate and a repeated legitimate event.
4. Why would `"Group"` and `" group "` create a modeling problem?
5. Explain how the Chi-square expected count is computed.
6. Why does a Chi-square association not prove causation?
7. For your assigned feature, defend your proposed cleaning decision.

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. Which data-quality problem in your sample would be most dangerous if left untreated?
2. Which suspicious value required the most judgment rather than a mechanical rule?
3. What did a plot reveal that the numerical summary did not?
4. Why is EDA performed before model training?
5. What should be documented when a row or value is changed or removed?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] your own student-ID-derived sample;
- [ ] dataset anatomy and variable-role analysis;
- [ ] mean/median interpretation;
- [ ] completed IQR function;
- [ ] histogram, boxplot, and scatter-plot interpretation;
- [ ] missing-value audit;
- [ ] duplicate analysis;
- [ ] completed invalid-value detector;
- [ ] category-cleaning function;
- [ ] manual expected-count calculation;
- [ ] completed expected-count function;
- [ ] completed Chi-square statistic;
- [ ] Chi-square interpretation;
- [ ] data-quality report;
- [ ] corrected Boolean debugging task;
- [ ] personalized feature analysis;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Data-analysis / preprocessing justification | **3** |
| Experimental and visual interpretation | **2** |
| Prediction / debugging / manual calculation | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require evidence that you can move from:

$$
\text{raw values}
\rightarrow
\text{diagnosis}
\rightarrow
\text{evidence}
\rightarrow
\text{justified treatment}.
$$

# Lab 2 Summary

You should now be able to perform a structured first investigation of a dataset:

$$
\boxed{
\text{Understand variables}
\rightarrow
\text{Summarize distributions}
\rightarrow
\text{Visualize}
\rightarrow
\text{Detect quality problems}
\rightarrow
\text{Justify cleaning decisions}
}
$$

### Key lessons

- Mean and median respond differently to extreme values.
- IQR rules flag unusual observations but do not decide whether they are errors.
- Missingness, duplicates, invalid values, and inconsistent categories are different problems.
- Plots complement summary statistics.
- Categorical association can be studied using contingency tables and Chi-square reasoning.
- EDA should produce **questions and justified actions**, not only plots.

**Next lab:** Preprocessing Pipelines and Feature Engineering.